# 01e — Feature Availability & Missingness Audit (Build 3, `v1`)

Quantifies, **on the frozen Build 1 cohort**: feature availability, invalid/missing values, baseline
timing relative to the plasma anchor, participant + event counts for each proposed model, and the
effect of unresolved cohort-quality scenarios on feasibility.

## This build measures. It does not decide.
- **No outcome model is fitted.** No C-index, calibration, hazard ratio, p-value or cutoff.
- **No imputation. No transformation. No standardisation.**
- **The frozen v1 cohort is not altered.** Scenarios are *descriptive recomputations* held in memory —
  no revised cohort is created or saved.
- **p-tau181 platforms are never combined**, converted, averaged, or prioritised.
- **No cognitive variable is selected.**
- Build 1 **and** Build 2 outputs are hashed before and after and asserted byte-identical.

## Reading this notebook
Availability findings (what the data *is*) are kept strictly separate from the scientific decisions
that remain open for the team. The report states which is which.

## 0. Setup, read-only guards, and hashes of everything already frozen

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 250)


def find_project_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "Data" / "raw").is_dir():
            return cand
    raise FileNotFoundError("Could not locate Data/raw above %s" % start)


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "Data" / "raw"
FREEZE_DIR = PROJECT_ROOT / "outputs" / "01c_mci_survival_cohort_freeze"

VERSION = "v1"
ALIGN_TOL_DAYS = 90            # locked
NARROW_TOL_DAYS = 30           # feasibility comparison only
MISSING_SENTINELS = [-4, -5]   # the EXISTING rule (plasma only)
CORE_ASSAYS = ["ptau217", "abeta42", "abeta40", "nfl", "gfap"]

# READ-ONLY. Never written by this notebook.
BUILD1_FILES = [
    "mci_survival_anchor_cohort_v1.tsv", "mci_survival_followup_cohort_v1.tsv",
    "mci_survival_primary_cohort_v1.tsv", "mci_survival_exclusion_log_v1.tsv",
    "mci_survival_cohort_flow_v1.tsv", "mci_survival_freeze_provenance_v1.json",
]
BUILD2_FILES = [
    "mci_survival_manual_qc_cases_v1.tsv", "mci_survival_manual_qc_longitudinal_context_v1.tsv",
    "mci_survival_manual_qc_form_v1.tsv", "mci_survival_manual_qc_sampling_summary_v1.tsv",
]
PROTECTED = BUILD1_FILES + BUILD2_FILES

ASSERTIONS: list[dict] = []


def check(name: str, ok, detail: str = "") -> None:
    ok = bool(ok)
    ASSERTIONS.append({"assertion": name, "passed": ok, "detail": detail})
    if not ok:
        raise AssertionError(f"ASSERTION FAILED: {name}\n  diagnostics: {detail}")
    print(f"  PASS  {name}" + (f"  [{detail}]" if detail else ""))


def rid_list(rids, limit: int = 15) -> str:
    rids = sorted(int(r) for r in rids)
    return f"n={len(rids)} RIDs=[{', '.join(str(r) for r in rids[:limit])}" \
           f"{', ...' if len(rids) > limit else ''}]"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def save_tsv(df: pd.DataFrame, name: str) -> Path:
    assert name not in PROTECTED, f"REFUSING to overwrite a protected Build 1/2 artifact: {name}"
    p = FREEZE_DIR / name
    df.to_csv(p, sep="\t", index=False)
    print(f"  saved -> {p.relative_to(PROJECT_ROOT)}   ({df.shape[0]} x {df.shape[1]})")
    return p


HASHES_BEFORE = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
print(f"Hashed {len(PROTECTED)} protected Build 1 + Build 2 artifacts (read-only).")
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")

Hashed 10 protected Build 1 + Build 2 artifacts (read-only).
python 3.14.0 | pandas 3.0.1 | numpy 2.4.3


## 1. Load the frozen cohorts *(authoritative participant definitions)*

Everything downstream is computed **from these files**. No participant is ever added, removed, or
re-derived.

In [2]:
BOOL_COLS = ["followup_eligible", "primary_eligible", "qc_dementia_on_or_before_anchor",
             "entry_age_available", "apoe4_available", "ptau217_available", "gfap_available",
             "nfl_available", "amyloid_ratio_available", "same_day_alignment",
             "entry_age_has_datestamp", "age_is_entry_age_fallback", "age_derivation_suspect",
             "broad_anchor_eligible", "nonpositive_followup_flag"]


def load_frozen(name):
    d = pd.read_csv(FREEZE_DIR / name, sep="\t")
    for c in BOOL_COLS:
        if c in d.columns:
            d[c] = d[c].astype("boolean").fillna(False).astype(bool)
    for c in [c for c in d.columns if c.endswith("_date")]:
        d[c] = pd.to_datetime(d[c], errors="coerce")
    return d


manifest = json.loads((FREEZE_DIR / f"mci_survival_freeze_provenance_{VERSION}.json").read_text())
ANCHOR = load_frozen(f"mci_survival_anchor_cohort_{VERSION}.tsv")
SURV = load_frozen(f"mci_survival_followup_cohort_{VERSION}.tsv")
PRIM = load_frozen(f"mci_survival_primary_cohort_{VERSION}.tsv")
QC_CASES = pd.read_csv(FREEZE_DIR / f"mci_survival_manual_qc_cases_{VERSION}.tsv", sep="\t")

for rec in manifest["outputs"]:
    fn = Path(rec["file"]).name
    check(f"[integrity] {fn} still matches its Build 1 manifest hash",
          sha256(FREEZE_DIR / fn) == rec["sha256"])

N_ANCHOR, N_SURV, N_PRIM = len(ANCHOR), len(SURV), len(PRIM)
EV_SURV = int((SURV["event_indicator"] == 1).sum())
EV_PRIM = int((PRIM["event_indicator"] == 1).sum())

# ---- ASSERTIONS 1 & 2 ----
check("frozen SURVIVAL cohort is still 410 participants / 86 events",
      (N_SURV, EV_SURV) == (410, 86), f"got {N_SURV}/{EV_SURV}")
check("frozen PRIMARY cohort is still 401 participants / 85 events",
      (N_PRIM, EV_PRIM) == (401, 85), f"got {N_PRIM}/{EV_PRIM}")
check("cohorts are nested: primary subset-of survival subset-of anchor",
      set(PRIM["RID"]) <= set(SURV["RID"]) <= set(ANCHOR["RID"]))

SURV_RIDS = set(SURV["RID"])
PRIM_RIDS = set(PRIM["RID"])
IS_EVENT = SURV.set_index("RID")["event_indicator"].eq(1)
print(f"\nFrozen: A={N_ANCHOR}  B={N_SURV} ({EV_SURV} events)  C={N_PRIM} ({EV_PRIM} events)")

  PASS  [integrity] mci_survival_anchor_cohort_v1.tsv still matches its Build 1 manifest hash
  PASS  [integrity] mci_survival_followup_cohort_v1.tsv still matches its Build 1 manifest hash
  PASS  [integrity] mci_survival_primary_cohort_v1.tsv still matches its Build 1 manifest hash
  PASS  [integrity] mci_survival_exclusion_log_v1.tsv still matches its Build 1 manifest hash
  PASS  [integrity] mci_survival_cohort_flow_v1.tsv still matches its Build 1 manifest hash
  PASS  frozen SURVIVAL cohort is still 410 participants / 86 events  [got 410/86]
  PASS  frozen PRIMARY cohort is still 401 participants / 85 events  [got 401/85]
  PASS  cohorts are nested: primary subset-of survival subset-of anchor

Frozen: A=535  B=410 (86 events)  C=401 (85 events)


## 2. Raw sources needed for the audit

- **plasma** — to count raw sentinels / non-positive values *at the selected anchor draw*.
- **CDR / MMSE / MOCA / FAQ** — to recompute cognition availability under **alternative timing rules**
  (feasibility only; frozen values are never modified).
- **FNIHBC trajectories** — p-tau181, **kept strictly per-platform**.
- `CDMEMORY` / `CDJUDGE` (CDR box scores) — the only **memory** and **executive-proxy** candidates that
  exist as single documented fields in the available data.

In [3]:
plasma_raw = pd.read_csv(RAW_DIR / "All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_05Mar2026.csv")
plasma_raw["plasma_src_row"] = np.arange(len(plasma_raw), dtype="int64")

cdr = pd.read_csv(RAW_DIR / "All_Subjects_CDR_05Mar2026.csv",
                  usecols=["RID", "VISDATE", "CDVERSION", "CDRSB", "CDMEMORY", "CDJUDGE"])
mmse = pd.read_csv(RAW_DIR / "All_Subjects_MMSE_05Mar2026.csv", usecols=["RID", "VISDATE", "MMSCORE"])
moca = pd.read_csv(RAW_DIR / "All_Subjects_MOCA_05Mar2026.csv", usecols=["RID", "VISDATE", "MOCA"])
faq = pd.read_csv(RAW_DIR / "All_Subjects_FAQ_05Mar2026.csv", usecols=["RID", "VISDATE", "FAQTOTAL"])
fnih = pd.read_csv(RAW_DIR / "All_Subjects_FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES_05Mar2026.csv",
                   usecols=["RID", "EXAMDATE", "VISCODE2", "PLASMA_BIOMARKER", "TESTNAME",
                            "ASSAYPLATFORM", "TESTVALUE", "UNITS"])

CDR_V = cdr[cdr["CDVERSION"].isin([1, 2])].copy()      # same filter as Build 1

# Cognitive candidates. Tier 1 = frozen pipeline. Tier 2 = audit-only (NOT in the frozen cohort).
COG = {
    "CDRSB":    dict(src=CDR_V, table="CDR",  field="CDRSB",    tier=1, domain="global severity / staging",
                     unit="0-18 (sum of boxes)"),
    "MMSCORE":  dict(src=mmse,  table="MMSE", field="MMSCORE",  tier=1, domain="global cognition",
                     unit="0-30"),
    "MOCA":     dict(src=moca,  table="MOCA", field="MOCA",     tier=1, domain="global cognition",
                     unit="0-30"),
    "FAQTOTAL": dict(src=faq,   table="FAQ",  field="FAQTOTAL", tier=1, domain="functional status",
                     unit="0-30"),
    "CDMEMORY": dict(src=CDR_V, table="CDR",  field="CDMEMORY", tier=2, domain="MEMORY (box score)",
                     unit="0-3 (box)"),
    "CDJUDGE":  dict(src=CDR_V, table="CDR",  field="CDJUDGE",  tier=2,
                     domain="EXECUTIVE proxy (judgement & problem-solving box)", unit="0-3 (box)"),
}
print("Cognitive candidates:")
for k, v in COG.items():
    print(f"  {k:9s} tier={v['tier']}  {v['table']:5s} {v['domain']}")
print("\nNOTE: no validated memory or executive COMPOSITE exists in the available raw data "
      "(no ADAS/RAVLT/UWNPSYCHSUM). CDMEMORY and CDJUDGE are single documented CDR box scores, "
      "audited here as tier-2 candidates. They are NOT in the frozen pipeline.")

Cognitive candidates:
  CDRSB     tier=1  CDR   global severity / staging
  MMSCORE   tier=1  MMSE  global cognition
  MOCA      tier=1  MOCA  global cognition
  FAQTOTAL  tier=1  FAQ   functional status
  CDMEMORY  tier=2  CDR   MEMORY (box score)
  CDJUDGE   tier=2  CDR   EXECUTIVE proxy (judgement & problem-solving box)

NOTE: no validated memory or executive COMPOSITE exists in the available raw data (no ADAS/RAVLT/UWNPSYCHSUM). CDMEMORY and CDJUDGE are single documented CDR box scores, audited here as tier-2 candidates. They are NOT in the frozen pipeline.


### 2b. Alternative timing rules for cognition *(feasibility only)*

The frozen pipeline aligns cognition with **nearest within ±90 days**. To answer "what would we have
if we demanded stricter timing?", we re-align the *same* raw sources under four rules. **The frozen
values are never modified** — assertion below proves rule 1 reproduces the frozen column exactly.

| Rule | Definition |
|---|---|
| `nearest_pm90` | nearest measurement within ±90 d — **the frozen rule** |
| `on_or_before_pm90` | nearest measurement in **[−90, 0]** (never after the anchor) |
| `within_pm30` | nearest measurement within ±30 d |
| `same_day_only` | measurement on the anchor date exactly |

In [4]:
ANCH = ANCHOR[["RID", "anchor_date"]].sort_values("anchor_date").reset_index(drop=True)


def clean_cog(spec) -> pd.DataFrame:
    """Exactly Build 1's align_scores source preparation."""
    s = spec["src"].copy()
    c = spec["field"]
    s["DATE"] = pd.to_datetime(s["VISDATE"], errors="coerce")
    s[c] = pd.to_numeric(s[c], errors="coerce")
    n_raw_dupe = int(s.dropna(subset=["DATE"]).duplicated(["RID", "DATE"], keep=False).sum())
    s = (s.dropna(subset=["DATE"]).dropna(subset=[c])
          .sort_values("DATE").drop_duplicates(["RID", "DATE"]))
    return s[["RID", "DATE", c]], n_raw_dupe


def align_rule(spec, rule: str) -> pd.DataFrame:
    """Return RID -> (value, date, offset) under one timing rule. Feasibility only."""
    c = spec["field"]
    s, _ = clean_cog(spec)
    if rule == "same_day_only":
        out = ANCH.merge(s.rename(columns={"DATE": "anchor_date"}), on=["RID", "anchor_date"],
                         how="left")
        out["_date"] = out["anchor_date"].where(out[c].notna())
    else:
        tol, direction = {
            "nearest_pm90": (ALIGN_TOL_DAYS, "nearest"),
            "on_or_before_pm90": (ALIGN_TOL_DAYS, "backward"),
            "within_pm30": (NARROW_TOL_DAYS, "nearest"),
        }[rule]
        out = pd.merge_asof(ANCH, s, left_on="anchor_date", right_on="DATE", by="RID",
                            tolerance=pd.Timedelta(days=tol), direction=direction)
        out["_date"] = out["DATE"]
    out["_offset"] = (out["_date"] - out["anchor_date"]).dt.days
    return out[["RID", c, "_date", "_offset"]].rename(
        columns={c: "value", "_date": "feature_date", "_offset": "offset_days"})


RULES = ["nearest_pm90", "on_or_before_pm90", "within_pm30", "same_day_only"]
COG_ALIGNED = {(k, r): align_rule(v, r) for k, v in COG.items() for r in RULES}

# ---- ASSERTION 7/14: rule 1 must reproduce the FROZEN cognition columns exactly ----
for k in ["CDRSB", "MMSCORE", "MOCA", "FAQTOTAL"]:
    a = COG_ALIGNED[(k, "nearest_pm90")].set_index("RID")["value"].sort_index()
    b = ANCHOR.set_index("RID")[k].sort_index()
    same = ((a - b).abs() <= 1e-9) | (a.isna() & b.isna())
    check(f"[no transformation] re-aligned {k} under the frozen rule reproduces the frozen column "
          f"exactly for all {N_ANCHOR} participants",
          bool(same.all()), rid_list(a.index[~same]))

print("\nCognition availability in the SURVIVAL cohort (n=410) by timing rule:")
hdr = f"  {'variable':10s} " + " ".join(f"{r:>18s}" for r in RULES)
print(hdr)
for k in COG:
    cells = []
    for r in RULES:
        v = COG_ALIGNED[(k, r)]
        n = int(v[v["RID"].isin(SURV_RIDS)]["value"].notna().sum())
        cells.append(f"{n:>18d}")
    print(f"  {k:10s} " + " ".join(cells))

  PASS  [no transformation] re-aligned CDRSB under the frozen rule reproduces the frozen column exactly for all 535 participants  [n=0 RIDs=[]]
  PASS  [no transformation] re-aligned MMSCORE under the frozen rule reproduces the frozen column exactly for all 535 participants  [n=0 RIDs=[]]
  PASS  [no transformation] re-aligned MOCA under the frozen rule reproduces the frozen column exactly for all 535 participants  [n=0 RIDs=[]]
  PASS  [no transformation] re-aligned FAQTOTAL under the frozen rule reproduces the frozen column exactly for all 535 participants  [n=0 RIDs=[]]

Cognition availability in the SURVIVAL cohort (n=410) by timing rule:
  variable         nearest_pm90  on_or_before_pm90        within_pm30      same_day_only
  CDRSB                     321                306                180                 75
  MMSCORE                   361                344                204                 79
  MOCA                      322                298                302             

## 3. p-tau181 — **per-platform** audit (never combined)

p-tau181 exists **only** in `FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES` (long format), on two platforms:

| Platform column | Assay |
|---|---|
| `QX_plasma_ptau181` | Quanterix / SIMOA |
| `Roche_plasma_ptau181` | Roche Elecsys |

**No cross-platform conversion, averaging, prioritisation, or imputation.** The two are reported and
modelled **separately**, and neither ever touches a frozen inclusion flag.

Two distinct availability notions are reported, because they answer different questions:
- **any measurement** (participant-level overlap with the sub-study);
- **anchor-aligned** — a measurement within ±90 d of the plasma anchor, which is what a *baseline*
  covariate actually requires.

In [5]:
PLATFORMS = {"ptau181_qx": ("QX_plasma_ptau181", "Quanterix / SIMOA"),
             "ptau181_roche": ("Roche_plasma_ptau181", "Roche Elecsys")}

fnih["DATE"] = pd.to_datetime(fnih["EXAMDATE"], errors="coerce")
fnih["value"] = pd.to_numeric(fnih["TESTVALUE"], errors="coerce")

p181_aligned = {}
for col, (biomarker, plat) in PLATFORMS.items():
    s = fnih[(fnih["PLASMA_BIOMARKER"] == biomarker) & fnih["value"].notna() & fnih["DATE"].notna()]
    s = s[["RID", "DATE", "value"]].sort_values("DATE").drop_duplicates(["RID", "DATE"])
    al = pd.merge_asof(ANCH, s, left_on="anchor_date", right_on="DATE", by="RID",
                       tolerance=pd.Timedelta(days=ALIGN_TOL_DAYS), direction="nearest")
    al["offset_days"] = (al["DATE"] - al["anchor_date"]).dt.days
    p181_aligned[col] = al[["RID", "value", "DATE", "offset_days"]].rename(
        columns={"value": col, "DATE": f"{col}_date", "offset_days": f"{col}_offset_days"})

ANY_P181 = {c: set(fnih.loc[(fnih["PLASMA_BIOMARKER"] == b) & fnih["value"].notna(), "RID"])
            for c, (b, _) in PLATFORMS.items()}

rows = []
for col, (biomarker, plat) in list(PLATFORMS.items()) + [
        ("ptau181_either", ("either platform", "union (reported only — NEVER a modeling variable)")),
        ("ptau181_both", ("both platforms", "intersection (reported only)"))]:
    if col == "ptau181_either":
        anyset = ANY_P181["ptau181_qx"] | ANY_P181["ptau181_roche"]
        aligned = None
    elif col == "ptau181_both":
        anyset = ANY_P181["ptau181_qx"] & ANY_P181["ptau181_roche"]
        aligned = None
    else:
        anyset = ANY_P181[col]
        aligned = p181_aligned[col]

    def _al(rids):
        if aligned is None:
            a1 = p181_aligned["ptau181_qx"].set_index("RID")["ptau181_qx"].notna()
            a2 = p181_aligned["ptau181_roche"].set_index("RID")["ptau181_roche"].notna()
            m = (a1 | a2) if col == "ptau181_either" else (a1 & a2)
            return set(m[m].index) & rids
        s = aligned.set_index("RID")[col].notna()
        return set(s[s].index) & rids

    al_surv, al_prim = _al(SURV_RIDS), _al(PRIM_RIDS)
    rows.append(dict(
        platform_column=col, source_file="All_Subjects_FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES_05Mar2026.csv",
        source_field=f"TESTVALUE where PLASMA_BIOMARKER=='{biomarker}'", assay_platform=plat,
        unit="pg/mL",
        n_participants_in_fnihbc=len(anyset),
        any_overlap_anchor_535=len(anyset & set(ANCHOR["RID"])),
        any_overlap_survival_410=len(anyset & SURV_RIDS),
        any_overlap_primary_401=len(anyset & PRIM_RIDS),
        anchor_aligned_pm90_survival=len(al_surv),
        anchor_aligned_pm90_survival_events=int(IS_EVENT.reindex(sorted(al_surv)).fillna(False).sum()),
        anchor_aligned_pm90_primary=len(al_prim),
        anchor_aligned_pm90_primary_events=int(IS_EVENT.reindex(sorted(al_prim)).fillna(False).sum()),
        combined_with_other_platform="NO — platforms kept strictly separate",
    ))

p181_tbl = pd.DataFrame(rows)
print(p181_tbl[["platform_column", "n_participants_in_fnihbc", "any_overlap_anchor_535",
                "any_overlap_survival_410", "any_overlap_primary_401",
                "anchor_aligned_pm90_survival", "anchor_aligned_pm90_survival_events",
                "anchor_aligned_pm90_primary", "anchor_aligned_pm90_primary_events"]].to_string(index=False))

# ---- ASSERTION 8 ----
check("p-tau181 platforms remain SEPARATE (no merged/averaged/prioritised modeling column)",
      "ptau181_qx" in p181_aligned and "ptau181_roche" in p181_aligned
      and not any(c.startswith("ptau181_combined") or c.startswith("ptau181_merged")
                  for c in list(p181_aligned)),
      "QX and Roche carried as independent columns; 'either'/'both' are REPORTED counts only")

platform_column  n_participants_in_fnihbc  any_overlap_anchor_535  any_overlap_survival_410  any_overlap_primary_401  anchor_aligned_pm90_survival  anchor_aligned_pm90_survival_events  anchor_aligned_pm90_primary  anchor_aligned_pm90_primary_events
     ptau181_qx                       355                      31                        25                       25                             1                                    1                            1                                   1
  ptau181_roche                       393                      39                        32                       32                             3                                    1                            3                                   1
 ptau181_either                       406                      40                        33                       33                             3                                    1                            3                                   1
   p

## 4. The audit frame

The frozen survival cohort **plus** audit-only columns (p-tau181 per platform, tier-2 cognition,
alternative-timing cognition). This frame exists **only in memory** for counting. It is **never**
written as a cohort and **never** feeds an inclusion flag.

In [6]:
AUD = SURV.copy()
for col in PLATFORMS:
    AUD = AUD.merge(p181_aligned[col], on="RID", how="left")

# every cognitive variable under every timing rule, as audit-only columns
for k in COG:
    for r in RULES:
        v = COG_ALIGNED[(k, r)].rename(columns={
            "value": f"{k}__{r}", "feature_date": f"{k}__{r}__date",
            "offset_days": f"{k}__{r}__offset"})
        AUD = AUD.merge(v, on="RID", how="left")

# ---- ASSERTION 5: no participant added ----
check("audit frame adds NO participant to the frozen survival cohort",
      set(AUD["RID"]) == SURV_RIDS and len(AUD) == N_SURV)
# ---- ASSERTION 6: no imputation ----
FROZEN_FEATS = ["entry_age", "APOE4_COUNT", "ptau217", "gfap", "nfl", "abeta42", "abeta40",
                "ratio_ab42_ab40", "CDRSB", "MMSCORE", "MOCA", "FAQTOTAL"]
check("no imputation: every frozen feature's missing-count in the audit frame equals the frozen file",
      all(int(AUD[c].isna().sum()) == int(SURV[c].isna().sum()) for c in FROZEN_FEATS),
      {c: int(SURV[c].isna().sum()) for c in FROZEN_FEATS}.__str__())
check("no transformation: every frozen feature's values are byte-equal to the frozen file",
      all(AUD.set_index("RID")[c].sort_index().equals(SURV.set_index("RID")[c].sort_index())
          for c in FROZEN_FEATS))
print(f"Audit frame: {AUD.shape[0]} x {AUD.shape[1]} (in memory only)")

  PASS  audit frame adds NO participant to the frozen survival cohort
  PASS  no imputation: every frozen feature's missing-count in the audit frame equals the frozen file  [{'entry_age': 0, 'APOE4_COUNT': 9, 'ptau217': 0, 'gfap': 43, 'nfl': 43, 'abeta42': 2, 'abeta40': 2, 'ratio_ab42_ab40': 2, 'CDRSB': 89, 'MMSCORE': 49, 'MOCA': 88, 'FAQTOTAL': 31}]
  PASS  no transformation: every frozen feature's values are byte-equal to the frozen file
Audit frame: 410 x 156 (in memory only)


## 5. Per-variable missingness and validity summary

In [7]:
# raw plasma rows that ARE the selected anchor draws (for sentinel / non-positive accounting)
anchor_rows = plasma_raw.set_index("plasma_src_row").loc[SURV["anchor_src_row"].to_numpy()]

PLASMA_MAP = {"ptau217": ("pT217_F", "Fujirebio", "pg/mL"),
              "gfap": ("GFAP_Q", "Quanterix", "pg/mL"),
              "nfl": ("NfL_Q", "Quanterix", "pg/mL"),
              "abeta42": ("AB42_F", "Fujirebio", "pg/mL"),
              "abeta40": ("AB40_F", "Fujirebio", "pg/mL")}

recs = []


def q(s, p):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(np.percentile(s, p)) if len(s) else np.nan


def stat_block(vals_surv, vals_prim):
    ev = IS_EVENT.reindex(vals_surv.index).fillna(False)
    av = vals_surv.notna()
    return dict(
        available_n_survival=int(av.sum()), missing_n_survival=int((~av).sum()),
        pct_available_survival=round(100 * av.mean(), 1),
        available_event_n=int((av & ev).sum()), missing_event_n=int((~av & ev).sum()),
        available_n_primary=int(vals_prim.notna().sum()),
        event_n_among_available_primary=int(
            (vals_prim.notna() & IS_EVENT.reindex(vals_prim.index).fillna(False)).sum()),
        n_zero_retained=int((vals_surv == 0).sum()), n_negative_retained=int((vals_surv < 0).sum()),
        n_infinite=int(np.isinf(pd.to_numeric(vals_surv, errors="coerce").fillna(0)).sum()),
        minimum=vals_surv.min(), q1=q(vals_surv, 25), median=vals_surv.median(),
        q3=q(vals_surv, 75), maximum=vals_surv.max(),
        n_duplicated_participant_values=int(vals_surv.index.duplicated().sum()),
    )


S = AUD.set_index("RID")
P = PRIM.set_index("RID")

# ---- primary: entry_age ----
recs.append(dict(canonical_column="entry_age", variable_name="Age at study entry", role="PRIMARY",
                 source_table="My_Table", source_field="entry_age", assay_platform="",
                 unit="years", data_type="float",
                 n_sentinel_in_source_table=int(pd.to_numeric(
                     pd.read_csv(RAW_DIR / "All_Subjects_My_Table_05Mar2026.csv")["entry_age"],
                     errors="coerce").isin(MISSING_SENTINELS).sum()),
                 n_negative_in_source_table=0, n_sentinel_removed=0, n_nonpositive_removed=0,
                 n_invalid_removed_total=0, n_undefined_ratio=0,
                 timing_source="NONE — My_Table carries no datestamp",
                 timing_available="NO — age-at-anchor is NOT reconstructable; no offset is fabricated",
                 **stat_block(S["entry_age"], P["entry_age"])))

# ---- primary: APOE4 ----
recs.append(dict(canonical_column="APOE4_COUNT", variable_name="APOE e4 allele count", role="PRIMARY",
                 source_table="APOERES", source_field="GENOTYPE -> count of '4'", assay_platform="",
                 unit="0-2 alleles", data_type="float(int-valued)",
                 n_sentinel_in_source_table=0, n_negative_in_source_table=0, n_sentinel_removed=0,
                 n_nonpositive_removed=0, n_invalid_removed_total=0, n_undefined_ratio=0,
                 timing_source="time-invariant (germline genotype)",
                 timing_available="N/A — time-invariant",
                 **stat_block(S["APOE4_COUNT"], P["APOE4_COUNT"])))

# ---- plasma biomarkers ----
for col, (raw_col, plat, unit) in PLASMA_MAP.items():
    rawv = pd.to_numeric(anchor_rows[raw_col], errors="coerce")
    n_sent = int(rawv.isin(MISSING_SENTINELS).sum())
    n_np = int(((~rawv.isin(MISSING_SENTINELS)) & rawv.notna() & (rawv <= 0)).sum())
    full = pd.to_numeric(plasma_raw[raw_col], errors="coerce")
    recs.append(dict(canonical_column=col, variable_name=raw_col,
                     role="PRIMARY" if col == "ptau217" else "SECONDARY",
                     source_table="UPENN_PLASMA_FUJIREBIO_QUANTERIX", source_field=raw_col,
                     assay_platform=plat, unit=unit, data_type="float",
                     n_sentinel_in_source_table=int(full.isin(MISSING_SENTINELS).sum()),
                     n_negative_in_source_table=int((full < 0).sum()),
                     n_sentinel_removed=n_sent, n_nonpositive_removed=n_np,
                     n_invalid_removed_total=n_sent + n_np, n_undefined_ratio=0,
                     timing_source="plasma anchor draw (the anchor row itself)",
                     timing_available="YES — offset = 0 by construction",
                     **stat_block(S[col], P[col])))

# ---- Abeta ratio ----
den = S["abeta40"]
recs.append(dict(canonical_column="ratio_ab42_ab40", variable_name="Ab42/Ab40 ratio (recomputed)",
                 role="SECONDARY", source_table="UPENN_PLASMA_FUJIREBIO_QUANTERIX",
                 source_field="AB42_F / AB40_F", assay_platform="Fujirebio (both components)",
                 unit="ratio (dimensionless)", data_type="float",
                 n_sentinel_in_source_table=0, n_negative_in_source_table=0, n_sentinel_removed=0,
                 n_nonpositive_removed=0, n_invalid_removed_total=0,
                 n_undefined_ratio=int((S["abeta42"].notna() & den.notna() & (den <= 0)).sum()),
                 timing_source="plasma anchor draw (both components from the SAME raw row)",
                 timing_available="YES — offset = 0 by construction",
                 **stat_block(S["ratio_ab42_ab40"], P["ratio_ab42_ab40"])))

# ---- p-tau181 per platform (audit-only, anchor-aligned) ----
for col, (biomarker, plat) in PLATFORMS.items():
    pv = S[col]
    pp = pd.Series(index=P.index, dtype=float)
    m = p181_aligned[col].set_index("RID")[col]
    pp = m.reindex(P.index)
    recs.append(dict(canonical_column=col, variable_name=f"p-tau181 ({plat})",
                     role="SECONDARY (NOT in frozen pipeline)",
                     source_table="FNIHBC_BLOOD_BIOMARKER_TRAJECTORIES",
                     source_field=f"TESTVALUE where PLASMA_BIOMARKER=='{biomarker}'",
                     assay_platform=plat, unit="pg/mL", data_type="float",
                     n_sentinel_in_source_table=0, n_negative_in_source_table=0, n_sentinel_removed=0,
                     n_nonpositive_removed=0, n_invalid_removed_total=0, n_undefined_ratio=0,
                     timing_source="FNIHBC EXAMDATE, nearest within +/-90d of the anchor",
                     timing_available="YES (dated)",
                     **stat_block(pv, pp)))

# ---- cognition (frozen alignment) ----
for k, spec in COG.items():
    _, n_dupe = clean_cog(spec)
    full = pd.to_numeric(spec["src"][spec["field"]], errors="coerce")
    colname = k if spec["tier"] == 1 else f"{k}__nearest_pm90"
    sv = S[colname] if colname in S.columns else S[f"{k}__nearest_pm90"]
    pv = COG_ALIGNED[(k, "nearest_pm90")].set_index("RID")["value"].reindex(P.index)
    off = S[f"{k}__nearest_pm90__offset"]
    recs.append(dict(canonical_column=(k if spec["tier"] == 1 else f"{k} (audit-only)"),
                     variable_name=f"{spec['field']} — {spec['domain']}",
                     role=("SECONDARY / cognition" if spec["tier"] == 1
                           else "CANDIDATE (tier-2, NOT in frozen pipeline)"),
                     source_table=spec["table"], source_field=spec["field"], assay_platform="",
                     unit=spec["unit"], data_type="float",
                     n_sentinel_in_source_table=int(full.isin(MISSING_SENTINELS).sum()),
                     n_negative_in_source_table=int((full < 0).sum()),
                     n_sentinel_removed=0, n_nonpositive_removed=0, n_invalid_removed_total=0,
                     n_undefined_ratio=0,
                     timing_source=f"{spec['table']}.VISDATE, nearest within +/-90d of the anchor",
                     timing_available="YES (dated)",
                     n_duplicated_source_rows_collapsed=n_dupe,
                     **{**stat_block(sv, pv),
                        "n_post_anchor_offset": int((off > 0).sum()),
                        "max_positive_offset_days": (float(off[off > 0].max()) if (off > 0).any() else np.nan)}))

missingness = pd.DataFrame(recs)
COLORD = ["canonical_column", "variable_name", "role", "source_table", "source_field",
          "assay_platform", "unit", "data_type",
          "available_n_survival", "missing_n_survival", "pct_available_survival",
          "available_event_n", "missing_event_n",
          "available_n_primary", "event_n_among_available_primary",
          "n_sentinel_in_source_table", "n_negative_in_source_table",
          "n_sentinel_removed", "n_nonpositive_removed", "n_invalid_removed_total",
          "n_zero_retained", "n_negative_retained", "n_infinite", "n_undefined_ratio",
          "minimum", "q1", "median", "q3", "maximum",
          "n_duplicated_participant_values", "n_duplicated_source_rows_collapsed",
          "n_post_anchor_offset", "max_positive_offset_days",
          "timing_source", "timing_available"]
missingness = missingness.reindex(columns=COLORD)
print(missingness[["canonical_column", "role", "available_n_survival", "pct_available_survival",
                   "available_event_n", "available_n_primary", "n_negative_in_source_table",
                   "n_negative_retained", "n_zero_retained"]].to_string(index=False))

     canonical_column                                       role  available_n_survival  pct_available_survival  available_event_n  available_n_primary  n_negative_in_source_table  n_negative_retained  n_zero_retained
            entry_age                                    PRIMARY                   410                   100.0                 86                  401                           0                    0                0
          APOE4_COUNT                                    PRIMARY                   401                    97.8                 85                  401                           0                    0              215
              ptau217                                    PRIMARY                   410                   100.0                 86                  401                           0                    0                0
                 gfap                                  SECONDARY                   367                    89.5                 86   

## 6. Aβ42/Aβ40 ratio audit

`ratio_ab42_ab40 = AB42_F / AB40_F`. Both components come from **the same raw plasma row** — the
selected anchor draw — so participant, draw, date and platform (Fujirebio) are **identical by
construction**; a mismatch is structurally impossible. The vendor's own `AB42_AB40_F` is preserved
separately and compared as a QC check (never substituted).

In [8]:
n42 = int(S["abeta42"].notna().sum())
n40 = int(S["abeta40"].notna().sum())
both = S["abeta42"].notna() & S["abeta40"].notna()
den_zero = int((S["abeta40"] == 0).sum())
den_nonpos = int((S["abeta40"] <= 0).sum())
valid_ratio = S["ratio_ab42_ab40"].notna()
ev = IS_EVENT.reindex(S.index).fillna(False)

# vendor vs recomputed (QC only)
cmpv = S[["ratio_ab42_ab40", "ratio_ab42_ab40_vendor"]].dropna()
maxdiff = float((cmpv["ratio_ab42_ab40"] - cmpv["ratio_ab42_ab40_vendor"]).abs().max()) if len(cmpv) else np.nan

ratio_audit = pd.DataFrame([
    ("formula", "ratio_ab42_ab40 = abeta42 / abeta40  (raw AB42_F / AB40_F)"),
    ("numerator field", "AB42_F (Fujirebio)"),
    ("denominator field", "AB40_F (Fujirebio)"),
    ("same participant / draw / date / platform?",
     "YES — both are columns of the SAME raw plasma row (the anchor draw); mismatch is impossible"),
    ("N with Ab42 valid (survival)", n42),
    ("N with Ab40 valid (survival)", n40),
    ("N with BOTH components valid", int(both.sum())),
    ("N with denominator == 0", den_zero),
    ("N with non-positive denominator", den_nonpos),
    ("N with mismatched date or platform", 0),
    ("N with a valid derived ratio", int(valid_ratio.sum())),
    ("N undefined / infinite ratios", int(np.isinf(S["ratio_ab42_ab40"].fillna(0)).sum())),
    ("Events among valid-ratio participants", int((valid_ratio & ev).sum())),
    ("Vendor AB42_AB40_F preserved separately?", "YES (ratio_ab42_ab40_vendor)"),
    ("max |recomputed - vendor| (QC only)", f"{maxdiff:.2e}"),
], columns=["item", "value"])
print(ratio_audit.to_string(index=False))

# ---- ASSERTION 9 ----
expected = S["abeta42"].notna() & S["abeta40"].notna() & (S["abeta40"] > 0)
check("every derived Ab42/Ab40 ratio uses compatible, valid components "
      "(non-null iff both components valid and denominator > 0, from the same raw row)",
      bool((valid_ratio == expected).all()),
      rid_list(S.index[valid_ratio != expected]))
check("raw Ab42 and Ab40 are preserved alongside the ratio",
      {"abeta42", "abeta40", "ratio_ab42_ab40_vendor"} <= set(SURV.columns))

                                      item                                                                                       value
                                   formula                                  ratio_ab42_ab40 = abeta42 / abeta40  (raw AB42_F / AB40_F)
                           numerator field                                                                          AB42_F (Fujirebio)
                         denominator field                                                                          AB40_F (Fujirebio)
same participant / draw / date / platform? YES — both are columns of the SAME raw plasma row (the anchor draw); mismatch is impossible
              N with Ab42 valid (survival)                                                                                         408
              N with Ab40 valid (survival)                                                                                         408
              N with BOTH components valid             

## 7. Feature timing audit

`feature_date − plasma_anchor_date`, bucketed. `entry_age` is **undated** — no offset is fabricated.
Plasma biomarkers sit **on** the anchor draw (offset 0 by construction). Cognition is the only feature
family where timing is a live question.

In [9]:
BUCKETS = [("before -90d", lambda o: o < -90), ("-90 to -31d", lambda o: (o >= -90) & (o <= -31)),
           ("-30 to -1d", lambda o: (o >= -30) & (o <= -1)), ("same day (0)", lambda o: o == 0),
           ("+1 to +30d", lambda o: (o >= 1) & (o <= 30)), ("+31 to +60d", lambda o: (o >= 31) & (o <= 60)),
           ("+61 to +90d", lambda o: (o >= 61) & (o <= 90)), ("after +90d", lambda o: o > 90)]

trows = []


def timing_row(name, role, offsets, n_universe, timing_source, timing_available, extra=None):
    o = pd.to_numeric(offsets, errors="coerce")
    r = dict(feature=name, role=role, timing_source=timing_source, timing_available=timing_available,
             n_universe=n_universe, n_dated=int(o.notna().sum()), n_undated_or_missing=int(o.isna().sum()))
    for lab, f in BUCKETS:
        r[lab] = int(f(o.dropna()).sum())
    r["n_post_anchor_positive_offset"] = int((o > 0).sum())
    r["max_positive_offset_days"] = float(o[o > 0].max()) if (o > 0).any() else np.nan
    r["min_offset_days"] = float(o.min()) if o.notna().any() else np.nan
    r["max_offset_days"] = float(o.max()) if o.notna().any() else np.nan
    r["potential_post_anchor_leakage"] = "YES — review" if (o > 0).any() else "no"
    if extra:
        r.update(extra)
    trows.append(r)


timing_row("entry_age", "PRIMARY", pd.Series([np.nan] * N_SURV), N_SURV,
           "NONE — My_Table has no datestamp",
           "NO — age-at-anchor cannot be reconstructed; NO offset fabricated")
timing_row("APOE4_COUNT", "PRIMARY", pd.Series([np.nan] * N_SURV), N_SURV,
           "time-invariant (germline)", "N/A — time-invariant")
for col in ["ptau217", "gfap", "nfl", "abeta42", "abeta40", "ratio_ab42_ab40"]:
    o = pd.Series(np.where(S[col].notna(), 0.0, np.nan), index=S.index)
    timing_row(col, "PRIMARY" if col == "ptau217" else "SECONDARY", o, N_SURV,
               "plasma anchor draw", "YES — offset = 0 by construction")
for col in PLATFORMS:
    timing_row(col, "SECONDARY (audit-only)", S[f"{col}_offset_days"], N_SURV,
               "FNIHBC EXAMDATE, nearest +/-90d", "YES (dated)")
for k, spec in COG.items():
    timing_row(k, f"cognition tier-{spec['tier']} — {spec['domain']}",
               S[f"{k}__nearest_pm90__offset"], N_SURV,
               f"{spec['table']}.VISDATE, nearest +/-90d (FROZEN rule)", "YES (dated)")

feature_timing = pd.DataFrame(trows)
print(feature_timing[["feature", "n_dated", "n_undated_or_missing", "-90 to -31d", "-30 to -1d",
                      "same day (0)", "+1 to +30d", "+31 to +60d", "+61 to +90d",
                      "n_post_anchor_positive_offset", "max_positive_offset_days"]].to_string(index=False))

# ---- ASSERTIONS 10, 11, 12 ----
for k in COG:
    o, d = S[f"{k}__nearest_pm90__offset"], S[f"{k}__nearest_pm90__date"]
    check(f"[timing] every {k} offset traces to a feature date AND the anchor date",
          bool((o.notna() == d.notna()).all())
          and bool(((d - S["anchor_date"]).dt.days.dropna() == o.dropna()).all()))
check("entry_age is NOT assigned a fabricated date or offset",
      bool(ANCHOR["age_source_date"].isna().all())
      and not bool(ANCHOR["entry_age_has_datestamp"].any())
      and bool(feature_timing.loc[feature_timing["feature"] == "entry_age", "n_dated"].iloc[0] == 0))
_post = {k: int((S[f"{k}__nearest_pm90__offset"] > 0).sum()) for k in COG}
check("post-anchor cognition is identified EXPLICITLY (reported, not silently dropped)",
      all(v >= 0 for v in _post.values()), f"post-anchor counts: {_post}")

        feature  n_dated  n_undated_or_missing  -90 to -31d  -30 to -1d  same day (0)  +1 to +30d  +31 to +60d  +61 to +90d  n_post_anchor_positive_offset  max_positive_offset_days
      entry_age        0                   410            0           0             0           0            0            0                              0                       NaN
    APOE4_COUNT        0                   410            0           0             0           0            0            0                              0                       NaN
        ptau217      410                     0            0           0           410           0            0            0                              0                       NaN
           gfap      367                    43            0           0           367           0            0            0                              0                       NaN
            nfl      367                    43            0           0           367          

## 8. Cognitive timing feasibility — availability under each rule

**Feasibility comparison only.** No timing rule is chosen, and the frozen cognition values are
untouched.

In [10]:
crows = []
for k, spec in COG.items():
    for r in RULES:
        v = COG_ALIGNED[(k, r)].set_index("RID")["value"]
        sv, pv = v.reindex(sorted(SURV_RIDS)), v.reindex(sorted(PRIM_RIDS))
        evs = IS_EVENT.reindex(sv.index).fillna(False)
        evp = IS_EVENT.reindex(pv.index).fillna(False)
        off = COG_ALIGNED[(k, r)].set_index("RID")["offset_days"].reindex(sorted(SURV_RIDS))
        crows.append(dict(
            variable=k, tier=spec["tier"], domain=spec["domain"], timing_rule=r,
            is_frozen_rule=(r == "nearest_pm90"),
            available_n_survival=int(sv.notna().sum()),
            pct_of_survival=round(100 * sv.notna().mean(), 1),
            available_events_survival=int((sv.notna() & evs).sum()),
            available_n_primary=int(pv.notna().sum()),
            available_events_primary=int((pv.notna() & evp).sum()),
            n_post_anchor=int((off > 0).sum()),
            max_positive_offset=(float(off[off > 0].max()) if (off > 0).any() else np.nan)))

cog_timing = pd.DataFrame(crows)
print(cog_timing[["variable", "tier", "timing_rule", "available_n_survival", "pct_of_survival",
                  "available_events_survival", "available_n_primary", "n_post_anchor"]].to_string(index=False))
feature_timing_out = pd.concat(
    [feature_timing.assign(section="offset_distribution"),
     cog_timing.assign(section="cognitive_timing_rule_feasibility")], ignore_index=True)

variable  tier       timing_rule  available_n_survival  pct_of_survival  available_events_survival  available_n_primary  n_post_anchor
   CDRSB     1      nearest_pm90                   321             78.3                         51                  314             15
   CDRSB     1 on_or_before_pm90                   306             74.6                         51                  299              0
   CDRSB     1       within_pm30                   180             43.9                         26                  178              7
   CDRSB     1     same_day_only                    75             18.3                          3                   75              0
 MMSCORE     1      nearest_pm90                   361             88.0                         82                  354             17
 MMSCORE     1 on_or_before_pm90                   344             83.9                         82                  337              0
 MMSCORE     1       within_pm30                   204 

## 9. Model-complete cohort counts

Universe = the **frozen survival cohort (410)**. A participant is model-complete when **every**
required variable is nonmissing. **No model is fitted.**

In [11]:
FU = S["time_to_event_or_censor_days"]
EVS = IS_EVENT.reindex(S.index).fillna(False)


def model_row(mid, label, req_cols, timing_rule="n/a", note=""):
    ok = pd.Series(True, index=S.index)
    lost = {}
    for c in req_cols:
        present = S[c].notna()
        lost[c] = int((~present).sum())
        ok &= present
    n = int(ok.sum())
    e = int((ok & EVS).sum())
    t = FU[ok]
    return dict(
        model_id=mid, model_label=label, required_variables="|".join(req_cols),
        timing_rule=timing_rule, complete_case_n=n, event_n=e, censored_n=n - e,
        pct_of_survival_410=round(100 * n / N_SURV, 1),
        n_lost_total=N_SURV - n,
        n_lost_by_variable="|".join(f"{k}={v}" for k, v in lost.items()),
        median_followup_days=(round(float(t.median()), 1) if n else np.nan),
        q1_followup_days=(round(float(np.percentile(t.dropna(), 25)), 1) if n else np.nan),
        q3_followup_days=(round(float(np.percentile(t.dropna(), 75)), 1) if n else np.nan),
        n_fu_le_30=int((t <= 30).sum()), n_fu_le_90=int((t <= 90).sum()),
        n_fu_le_180=int((t <= 180).sum()), n_fu_le_365=int((t <= 365).sum()),
        notes=note)


BASE = ["entry_age", "APOE4_COUNT"]
mrows = [
    model_row("M1", "age + APOE4", BASE),
    model_row("M2", "age + APOE4 + p-tau217", BASE + ["ptau217"],
              note="PRIMARY MODEL — must equal the frozen primary cohort"),
    model_row("M3", "age + APOE4 + p-tau181 (Quanterix)", BASE + ["ptau181_qx"],
              note="p-tau181 NOT in frozen pipeline; platforms never combined"),
    model_row("M4", "age + APOE4 + p-tau181 (Roche)", BASE + ["ptau181_roche"],
              note="p-tau181 NOT in frozen pipeline; platforms never combined"),
    model_row("M5", "age + APOE4 + p-tau217 + GFAP", BASE + ["ptau217", "gfap"]),
    model_row("M6", "age + APOE4 + p-tau217 + Ab42/Ab40", BASE + ["ptau217", "ratio_ab42_ab40"]),
]
for k, spec in COG.items():
    if spec["tier"] != 1:
        continue
    for r in RULES:
        mrows.append(model_row(f"M7_{k}", f"age + APOE4 + {k}", BASE + [f"{k}__{r}"], timing_rule=r))
        mrows.append(model_row(f"M8_{k}", f"age + APOE4 + p-tau217 + {k}",
                               BASE + ["ptau217", f"{k}__{r}"], timing_rule=r))
for r in RULES:
    mrows.append(model_row("M9_mem_exec", "age + APOE4 + p-tau217 + CDMEMORY + CDJUDGE",
                           BASE + ["ptau217", f"CDMEMORY__{r}", f"CDJUDGE__{r}"], timing_rule=r,
                           note="TIER-2 candidates, NOT in the frozen pipeline; CDJUDGE is an "
                                "executive PROXY, not a validated executive composite"))

models = pd.DataFrame(mrows)

# ---- ASSERTIONS 3, 14, 15 ----
m2 = models[models["model_id"] == "M2"].iloc[0]
check("M2 (age + APOE4 + p-tau217) reproduces the frozen PRIMARY cohort counts exactly",
      (int(m2["complete_case_n"]), int(m2["event_n"])) == (N_PRIM, EV_PRIM),
      f"M2 = {int(m2['complete_case_n'])}/{int(m2['event_n'])}; frozen C = {N_PRIM}/{EV_PRIM}")
m2_ok = S["entry_age"].notna() & S["APOE4_COUNT"].notna() & S["ptau217"].notna()
check("M2 complete-case participant IDs EQUAL the frozen primary-cohort participant IDs",
      set(S.index[m2_ok]) == PRIM_RIDS,
      rid_list(set(S.index[m2_ok]) ^ PRIM_RIDS))
_indep = int((SURV["entry_age"].notna() & SURV["APOE4_COUNT"].notna()
              & SURV["ptau217"].notna()).sum())
check("every model-complete count is reproducible from required-variable nonmissingness alone",
      _indep == int(m2["complete_case_n"]) == N_PRIM)

print(models[["model_id", "timing_rule", "complete_case_n", "event_n", "pct_of_survival_410",
              "n_lost_by_variable"]].to_string(index=False))

  PASS  M2 (age + APOE4 + p-tau217) reproduces the frozen PRIMARY cohort counts exactly  [M2 = 401/85; frozen C = 401/85]
  PASS  M2 complete-case participant IDs EQUAL the frozen primary-cohort participant IDs  [n=0 RIDs=[]]
  PASS  every model-complete count is reproducible from required-variable nonmissingness alone
   model_id       timing_rule  complete_case_n  event_n  pct_of_survival_410                                                                                 n_lost_by_variable
         M1               n/a              401       85                 97.8                                                                          entry_age=0|APOE4_COUNT=9
         M2               n/a              401       85                 97.8                                                                entry_age=0|APOE4_COUNT=9|ptau217=0
         M3               n/a                1        1                  0.2                                                           entry_age=0|APOE

## 10. Unresolved cohort-quality scenarios

**Descriptive feasibility only.** The frozen v1 cohort stays authoritative; **no revised cohort is
created or saved**, and no frozen inclusion flag is touched.

In [12]:
FLAGS_SNAPSHOT = {c: SURV[c].copy() for c in
                  ["followup_eligible", "primary_eligible", "qc_dementia_on_or_before_anchor"]}

# recompute both flags for the FULL cohort straight from frozen columns
ANCHOR["_censor_is_anchor_visit"] = ((ANCHOR["event_indicator"] == 0)
                                     & ANCHOR["censor_src_row"].notna()
                                     & (ANCHOR["censor_src_row"] == ANCHOR["anchor_dx_src_row"]))
SURV["_censor_is_anchor_visit"] = ((SURV["event_indicator"] == 0)
                                   & SURV["censor_src_row"].notna()
                                   & (SURV["censor_src_row"] == SURV["anchor_dx_src_row"]))

# cross-check against the Build 2 packet (46 reviewed participants)
_q = QC_CASES.set_index("RID")["warn_censor_is_anchor_matching_visit"].astype(bool)
_mine = SURV.set_index("RID")["_censor_is_anchor_visit"].reindex(_q.index).fillna(False)
check("recomputed 'censor-is-anchor-visit' flag agrees with the Build 2 packet on all reviewed cases",
      bool((_q == _mine.reindex(_q.index).fillna(False)).all()))

PRE_DEM = set(ANCHOR.loc[ANCHOR["qc_dementia_on_or_before_anchor"], "RID"])
CIA = set(SURV.loc[SURV["_censor_is_anchor_visit"], "RID"])

srows = []


def scen(sid, label, stype, keep_anchor, keep_surv, threshold=np.nan, note=""):
    a = ANCHOR[ANCHOR["RID"].isin(keep_anchor)]
    b = SURV[SURV["RID"].isin(keep_surv)]
    c = b[b["primary_eligible"]]
    m2 = b[b["entry_age"].notna() & b["APOE4_COUNT"].notna() & b["ptau217"].notna()]
    srows.append(dict(
        scenario_id=sid, scenario_label=label, scenario_type=stype, threshold_days=threshold,
        anchor_n=len(a), anchor_delta=len(a) - N_ANCHOR,
        survival_n=len(b), survival_events=int((b["event_indicator"] == 1).sum()),
        survival_censored=int((b["event_indicator"] == 0).sum()),
        survival_delta_n=len(b) - N_SURV,
        survival_delta_events=int((b["event_indicator"] == 1).sum()) - EV_SURV,
        primary_n=len(c), primary_events=int((c["event_indicator"] == 1).sum()),
        primary_delta_n=len(c) - N_PRIM,
        primary_delta_events=int((c["event_indicator"] == 1).sum()) - EV_PRIM,
        m2_complete_n=len(m2), m2_events=int((m2["event_indicator"] == 1).sum()),
        n_removed=N_SURV - len(b), notes=note))


ALL_A, ALL_S = set(ANCHOR["RID"]), SURV_RIDS

scen("S1_frozen_v1", "Frozen v1 (authoritative)", "frozen", ALL_A, ALL_S,
     note="the cohort of record")
scen("S2_exclude_prior_dementia", "Exclude any dementia dx BEFORE the anchor", "exclusion",
     ALL_A - PRE_DEM, ALL_S - PRE_DEM,
     note="SCENARIO ONLY — not an adjudication; awaits Build 2 human review")

# ---- Scenario 3: minimum follow-up (outcome ascertained at T) ----
t_all = SURV.set_index("RID")["time_to_event_or_censor_days"]
ev_all = SURV.set_index("RID")["event_indicator"].eq(1)
for T in [30, 90, 180, 365, 730]:
    qualifies = ev_all | (t_all >= T)          # event at any time, OR event-free & observed to >= T
    keep = set(t_all.index[qualifies])
    removed = ALL_S - keep
    rem = SURV[SURV["RID"].isin(removed)]
    n_ev_removed = int((rem["event_indicator"] == 1).sum())
    scen(f"S3_minfu_{T}d", f"Outcome ascertained at >= {T} days", "min_followup", ALL_A, keep,
         threshold=T,
         note=(f"removed {len(removed)} ({int((rem['event_indicator']==0).sum())} censored, "
               f"{n_ev_removed} events) — removals are "
               f"{'EXCLUSIVELY censored' if n_ev_removed == 0 else 'NOT exclusively censored (!)'}"))

scen("S4_exclude_censor_is_anchor_visit",
     "Exclude participants censored at the anchor-matching MCI visit", "flag_exclusion",
     ALL_A, ALL_S - CIA,
     note="SCENARIO ONLY — these contribute <=90d of follow-up (median 12d)")

scenarios = pd.DataFrame(srows)

# ---- S4 overlaps ----
ov = []
fu30 = set(SURV.loc[SURV["time_to_event_or_censor_days"] <= 30, "RID"])
fu90 = set(SURV.loc[SURV["time_to_event_or_censor_days"] <= 90, "RID"])
for lab, other in [("follow-up <= 30 days", fu30), ("follow-up <= 90 days", fu90),
                   ("prior-dementia history", PRE_DEM & SURV_RIDS),
                   ("primary complete-case (C)", PRIM_RIDS)]:
    ov.append(dict(scenario_id="S4_overlap", scenario_label=f"censor-is-anchor-visit  AND  {lab}",
                   scenario_type="overlap", threshold_days=np.nan,
                   survival_n=len(CIA & other), notes=f"|CIA|={len(CIA)}, |{lab}|={len(other)}"))
scenarios = pd.concat([scenarios, pd.DataFrame(ov)], ignore_index=True)

print(scenarios[["scenario_id", "threshold_days", "anchor_n", "survival_n", "survival_events",
                 "primary_n", "primary_events", "m2_complete_n", "m2_events", "n_removed"]].to_string(index=False))

# ---- ASSERTION 13 ----
check("scenario computation did NOT alter any frozen inclusion flag",
      all(SURV[c].equals(FLAGS_SNAPSHOT[c]) for c in FLAGS_SNAPSHOT))
check("frozen cohorts are unchanged after all scenario work (still 410/86 and 401/85)",
      (len(SURV), int((SURV["event_indicator"] == 1).sum())) == (410, 86)
      and (len(PRIM), int((PRIM["event_indicator"] == 1).sum())) == (401, 85))
_bad = scenarios[(scenarios["scenario_type"] == "min_followup")
                 & scenarios["notes"].str.contains(r"NOT exclusively censored")]
check("min-follow-up scenarios remove ONLY censored participants (no event is ever dropped)",
      _bad.empty, "all events retained at every threshold")

  PASS  recomputed 'censor-is-anchor-visit' flag agrees with the Build 2 packet on all reviewed cases


                      scenario_id  threshold_days  anchor_n  survival_n  survival_events  primary_n  primary_events  m2_complete_n  m2_events  n_removed
                     S1_frozen_v1             NaN     535.0         410             86.0      401.0            85.0          401.0       85.0        0.0
        S2_exclude_prior_dementia             NaN     527.0         405             84.0      397.0            84.0          397.0       84.0        5.0
                     S3_minfu_30d            30.0     535.0         349             86.0      342.0            85.0          342.0       85.0       61.0
                     S3_minfu_90d            90.0     535.0         329             86.0      325.0            85.0          325.0       85.0       81.0
                    S3_minfu_180d           180.0     535.0         314             86.0      310.0            85.0          310.0       85.0       96.0
                    S3_minfu_365d           365.0     535.0         286           

## 11. Write outputs; verify Build 1 + Build 2 byte-identical

In [13]:
OUTPUTS = {
    f"mci_survival_feature_missingness_{VERSION}.tsv": missingness,
    f"mci_survival_model_complete_case_counts_{VERSION}.tsv": models,
    f"mci_survival_feature_timing_{VERSION}.tsv": feature_timing_out,
    f"mci_survival_feature_scenario_counts_{VERSION}.tsv": scenarios,
    f"mci_survival_ptau181_platform_availability_{VERSION}.tsv": p181_tbl,
}
paths = {n: save_tsv(d, n) for n, d in OUTPUTS.items()}

HASHES_AFTER = {f: sha256(FREEZE_DIR / f) for f in PROTECTED}
drift = [f for f in PROTECTED if HASHES_BEFORE[f] != HASHES_AFTER[f]]
check("ALL Build 1 and Build 2 artifacts are BYTE-IDENTICAL before and after Build 3",
      not drift, f"drifted: {drift}" if drift else f"{len(PROTECTED)} artifacts unchanged")

print("\nBuild 3 output hashes:")
for n, p in paths.items():
    print(f"  {n:52s} {sha256(p)[:16]}...")

  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_feature_missingness_v1.tsv   (16 x 35)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_model_complete_case_counts_v1.tsv   (42 x 18)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_feature_timing_v1.tsv   (40 x 33)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_feature_scenario_counts_v1.tsv   (12 x 19)
  saved -> outputs/01c_mci_survival_cohort_freeze/mci_survival_ptau181_platform_availability_v1.tsv   (4 x 14)
  PASS  ALL Build 1 and Build 2 artifacts are BYTE-IDENTICAL before and after Build 3  [10 artifacts unchanged]

Build 3 output hashes:
  mci_survival_feature_missingness_v1.tsv              cabe0fffaff0634b...
  mci_survival_model_complete_case_counts_v1.tsv       2d380fe33673dae3...
  mci_survival_feature_timing_v1.tsv                   b6611ed8a1426a1e...
  mci_survival_feature_scenario_counts_v1.tsv          e60b4a937a4f4e4f...
  mci_survival_ptau181_platform_availab

## 12. Build 3 summary

In [14]:
print("=" * 90)
print("BUILD 3 — FEATURE AVAILABILITY & MISSINGNESS AUDIT (v1)")
print("=" * 90)
print(f"Assertions : {sum(a['passed'] for a in ASSERTIONS)} passed / "
      f"{sum(not a['passed'] for a in ASSERTIONS)} failed  (of {len(ASSERTIONS)})")
print(f"Frozen cohort UNCHANGED: A={N_ANCHOR}  B={N_SURV} ({EV_SURV} ev)  C={N_PRIM} ({EV_PRIM} ev)")
print()
print("--- Primary model (M2 = age + APOE4 + p-tau217) ---")
print(f"    N={int(m2['complete_case_n'])}  events={int(m2['event_n'])}  "
      f"= the frozen primary cohort exactly")
print()
print("--- p-tau181 anchor-aligned availability (survival cohort, n=410) ---")
for _, r in p181_tbl.iterrows():
    print(f"    {r['platform_column']:16s} any={r['any_overlap_survival_410']:>3d}  "
          f"aligned+/-90d={r['anchor_aligned_pm90_survival']:>3d}  "
          f"events={r['anchor_aligned_pm90_survival_events']:>2d}")
print()
print("--- Cognition: post-anchor measurements under the FROZEN +/-90d rule ---")
for k in ["CDRSB", "MMSCORE", "MOCA", "FAQTOTAL"]:
    o = S[f"{k}__nearest_pm90__offset"]
    print(f"    {k:9s} available={int(S[k].notna().sum()):>3d}/410  "
          f"post-anchor={int((o > 0).sum()):>3d}  max offset=+{int(o.max()) if o.notna().any() else 0}d")
print()
print("--- Scenarios (descriptive; frozen cohort untouched) ---")
for _, r in scenarios[scenarios["scenario_type"] != "overlap"].iterrows():
    print(f"    {r['scenario_id']:32s} B={int(r['survival_n']):>3d} "
          f"(ev {int(r['survival_events']):>2d})   M2={int(r['m2_complete_n']):>3d} "
          f"(ev {int(r['m2_events']):>2d})")
print("=" * 90)

BUILD 3 — FEATURE AVAILABILITY & MISSINGNESS AUDIT (v1)
Assertions : 34 passed / 0 failed  (of 34)
Frozen cohort UNCHANGED: A=535  B=410 (86 ev)  C=401 (85 ev)

--- Primary model (M2 = age + APOE4 + p-tau217) ---
    N=401  events=85  = the frozen primary cohort exactly

--- p-tau181 anchor-aligned availability (survival cohort, n=410) ---
    ptau181_qx       any= 25  aligned+/-90d=  1  events= 1
    ptau181_roche    any= 32  aligned+/-90d=  3  events= 1
    ptau181_either   any= 33  aligned+/-90d=  3  events= 1
    ptau181_both     any= 24  aligned+/-90d=  1  events= 1

--- Cognition: post-anchor measurements under the FROZEN +/-90d rule ---
    CDRSB     available=321/410  post-anchor= 15  max offset=+83d
    MMSCORE   available=361/410  post-anchor= 17  max offset=+83d
    MOCA      available=322/410  post-anchor= 24  max offset=+49d
    FAQTOTAL  available=379/410  post-anchor= 51  max offset=+88d

--- Scenarios (descriptive; frozen cohort untouched) ---
    S1_frozen_v1          